# Optimizer Example

Optimize column data types for improved analysis performance and reduced memory usage.

The optimization pipeline automatically detects columns that can be safely
represented using more appropriate pandas dtypes. Conversions are applied only
when they satisfy the configured detection thresholds, ensuring existing data is
preserved whenever possible.

The following optimizations are performed:

* BooleanTypePlugin converts IsActive and Verified to nullable boolean.
* DateTimeTypePlugin converts OrderDate to datetime64[ns].
* NumericTypePlugin converts numeric strings in Age and Salary to numeric dtypes.
* CategoryTypePlugin converts low-cardinality text columns Department and Region to category, while leaving the high-cardinality Name column unchanged.

## Returns

**pandas.DataFrame**

A DataFrame with optimized column dtypes.

## Notes

An optimization report is stored in:

```python
df.attrs["danda_optimize_report"]
```

The same report is also available through:

```python
df.dg.report["optimize"]
```

## Examples

Create a DataFrame containing columns that can be optimized:

```python
>>> import pandas as pd

>>> df = pd.DataFrame({
...     "IsActive": ["True", "false", "1", "0", None],
...     "Verified": [1, 0, 1, 0, None],
...     "OrderDate": [
...         "2024-01-01",
...         "2024-02-15",
...         "2024-03-20",
...         None,
...         "2024-04-10",
...     ],
...     "Age": ["25", "30", "35", None, "40"],
...     "Salary": ["50000.5", "62000", "71000.25", "80000", None],
...     "Department": ["HR", "IT", "HR", "IT", "HR"],
...     "Region": ["East", "West", "East", "West", "East"],
...     "Name": ["Alice", "Bob", "Charlie", "David", "Eve"],
... })

>>> df.dtypes
IsActive      object
Verified      object
OrderDate     object
Age           object
Salary        object
Department    object
Region        object
Name          object
dtype: object
```

Optimize the DataFrame:

```python
>>> df = df.dg.optimize()

>>> df.dtypes
IsActive             boolean
Verified             boolean
OrderDate    datetime64[ns]
Age                 float64
Salary              float64
Department         category
Region             category
Name                 object
dtype: object
```

Inspect the optimization report:

```python
>>> df.dg.report["optimize"]
{
    "types": {
        "BooleanTypePlugin": [
            "IsActive",
            "Verified"
        ],
        "DateTimeTypePlugin": [
            "OrderDate"
        ],
        "NumericTypePlugin": [
            "Age",
            "Salary"
        ],
        "CategoryTypePlugin": [
            "Department",
            "Region"
        ]
    }
}
```

In this example:

* `IsActive` and `Verified` are converted to the nullable `boolean` dtype.
* `OrderDate` is converted to `datetime64[ns]`.
* `Age` and `Salary` are converted from numeric strings to numeric dtypes.
* `Department` and `Region` are converted to `category` because they contain a small number of unique values.
* `Name` remains unchanged because it has high cardinality and does not satisfy the criteria for categorical conversion.


In [1]:
import pandas as pd
# notice the need to danda import to have accessor df
import danda # noqa: F401  # registers the pandas accessor
from danda.report_renderer import ReportRenderer

renderer = ReportRenderer()

In [2]:
df = pd.DataFrame({
    # Boolean-like strings and integers
    "IsActive": ["True", "false", "1", "0", None],
    "Verified": [1, 0, 1, 0, None],

    # Datetime-like strings
    "OrderDate": [
        "2024-01-01",
        "2024-02-15",
        "2024-03-20",
        None,
        "2024-04-10",
    ],

    # Numeric strings
    "Age": ["25", "30", "35", None, "40"],
    "Salary": ["50000.5", "62000", "71000.25", "80000", None],

    # Low-cardinality text
    "Department": ["HR", "IT", "HR", "IT", "HR"],
    "Region": ["East", "West", "East", "West", "East"],

    # High-cardinality text (should remain object/string)
    "Name": ["Alice", "Bob", "Charlie", "David", "Eve"],
})

print(df.dtypes)
df

IsActive          str
Verified      float64
OrderDate         str
Age               str
Salary            str
Department        str
Region            str
Name              str
dtype: object


,IsActive,Verified,OrderDate,Age,Salary,Department,Region,Name
0,True,1.0,2024-01-01,25,50000.5,HR,East,Alice
1,false,0.0,2024-02-15,30,62000,IT,West,Bob
2,1,1.0,2024-03-20,35,71000.25,HR,East,Charlie
3,0,0.0,NaN,NaN,80000,IT,West,David
4,NaN,NaN,2024-04-10,40,NaN,HR,East,Eve


In [3]:
optimized_df = df.dg.optimize()

print(optimized_df.dtypes)
optimized_df

IsActive             boolean
Verified             float64
OrderDate     datetime64[us]
Age                  float64
Salary               float64
Department               str
Region                   str
Name                     str
dtype: object


,IsActive,Verified,OrderDate,Age,Salary,Department,Region,Name
0,True,1.0,2024-01-01,25.0,50000.50,HR,East,Alice
1,False,0.0,2024-02-15,30.0,62000.00,IT,West,Bob
2,True,1.0,2024-03-20,35.0,71000.25,HR,East,Charlie
3,False,0.0,NaT,NaN,80000.00,IT,West,David
4,<NA>,NaN,2024-04-10,40.0,NaN,HR,East,Eve


## Improve the configuration

The example above uses only five rows. With such a small dataset, the default
category_threshold (10%) is too strict to classify Department and Region
as categorical columns. Their unique-value ratio exceeds the default threshold,
so they remain as object columns.

You can lower or raise the detection threshold depending on your dataset. For
this small example, increasing the threshold to 0.5 (50%) allows both columns
to be recognized as categorical.

In [4]:
df.dg.config.types.category_threshold = .4

In [5]:
optimized_df1 = df.dg.optimize()

print(optimized_df1.dtypes)
optimized_df1

IsActive             boolean
Verified             float64
OrderDate     datetime64[us]
Age                  float64
Salary               float64
Department          category
Region              category
Name                     str
dtype: object


,IsActive,Verified,OrderDate,Age,Salary,Department,Region,Name
0,True,1.0,2024-01-01,25.0,50000.50,HR,East,Alice
1,False,0.0,2024-02-15,30.0,62000.00,IT,West,Bob
2,True,1.0,2024-03-20,35.0,71000.25,HR,East,Charlie
3,False,0.0,NaT,NaN,80000.00,IT,West,David
4,<NA>,NaN,2024-04-10,40.0,NaN,HR,East,Eve


In [6]:
# This will show what happen
print(renderer.render(optimized_df1.dg.report))

Danda Report

Optimize
--------

BooleanTypePlugin
    convert these columns to boolean IsActive

DateTimeTypePlugin
    Converted the following columns to datetime: OrderDate

NumericTypePlugin
    Converted the following columns to numeric: Age, Salary

CategoryTypePlugin
    Converted the following columns to category: Department, Region

Execution
---------
Plugins executed : 4
Plugin names     : BooleanTypePlugin, DateTimeTypePlugin, NumericTypePlugin, CategoryTypePlugin
Memory before    : 593 bytes
Memory after     : 421 bytes
Memory saved     : 172 bytes (29.0%)
